# Constructing Knowledge Graphs with LLMs

Relying solely on text embeddings can lead to chanllenges in scenarios where data needs to be structured to answer questions that require filtering, counting, or aggregation operations.

We need to transform these unstructured data into structured formats suitable for knowledge graph construction, using LLMs for automated data extraction.

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from pydantic import BaseModel, Field
from openai import OpenAI
from typing import Optional, List
import json

from utils.utils import neo4j_driver

client = OpenAI()

## Extracting Structured Data from Text

![basic vector retrieval](./imgs/basic-vec-retrieval-strategy.png)

The figure above illustrates how enterprise documents are broken down into text chunks and indexed using text embeddings.

When an end user asks a specific question, the system retrieves the most relevant chunks. However, if multiple documents contain different pieces of information relevant to the question, the retrieval process may unintentionally pull information from various documents, mixing relevant chunks from the target document with irrelevant ones from others.

This happens because the system focuses on retrieving *top-ranked* text chunks based on *similarity*, without always distinguishing whether the chunks come from the correct document.

*Text embeddings are primarily designed to retrieve semantically similar content, not to handle operations like filtering, sorting, or aggregating data.* To handle such operations, structured data is required, as text embeddings alone are not well-suited for these operations.

Using LLMs for structured data extraction is particularly useful when dealing with large volumes of documents where manually identifying and organizing such information would be labor intensive and time consuming.

![buliding knowledge graph with LLMs](./imgs/build-kg-from-text-with-llm.png)

The figure above illustrates the workflow of building a knowledge graph from unstructured text using LLMs.


### Structured Outputs Model Definition

Structured Outputs significantly simplifies the development process by ensuring that the LLM responses adhere to a predefined schema.

In [3]:
contract_types = [
    "Service Agreement",
    "Licensing Agreement",
    "Non-Disclosure Agreement (NDA)",
    "Partnership Agreement",
    "Lease Agreement"
]


class Location(BaseModel):
    """
    A data model representing a physical location including address, city, state, and country.
    """
    address: Optional[str] = Field(
        ..., 
        description="The street address of the location."
    )
    city: Optional[str] = Field(
        ..., 
        description="The city of the location."
    )
    state: Optional[str] = Field(
        ..., 
        description="The state or region of the location."
    )
    country: str = Field(
        ...,
        description="The country of the location. Use the two-letter ISO standard.",
    )


class Organization(BaseModel):
    """
    A data model representing an organization, including its name and location
    """
    name: str = Field(
        ...,
        description="The name of the organization."
    )
    location: Location = Field(
        ...,
        description="The primary location of the organization."
    )
    role: str = Field(
        ...,
        description="The role of the organization in the contract, such as 'provider', 'client', 'supplier', etc."
    )


In [6]:
class Contract(BaseModel):
    """
    A data model representing the key details of a contract
    """
    contract_type: str = Field(
        ...,
        description="The type of contract being entered into.",
        enum=contract_types
    )
    parties: List[Organization] = Field(
        ...,
        description="List of parties involved in the contract, with details of each party's role."
    )
    effective_date: str = Field(
        ...,
        description="The date when the contract becomes effective. Use yyyy-MM-dd format."
    )
    term: str = Field(
        ...,
        description="The duration of the contract, including provisions for renewal or termination if applicable."
    )
    contract_scope: str = Field(
        ...,
        description="Description of the scope of the contract, including rights, duties, and any limitations."
    )
    end_date: Optional[str] = Field(
        ...,
        description="The date when the contract becomes expired. Use yyyy-MM-dd format."
    )
    total_amount: Optional[float] = Field(
        ...,
        description="Total value of the contract."
    )
    governing_law: Optional[Location] = Field(
        ...,
        description="The jurisdiction's laws governing the contract."
    )

C:\Users\binli\AppData\Local\Temp\ipykernel_15372\3260930869.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'enum'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  contract_type: str = Field(


### Structured Outputs Extraction Request

In [7]:
sys_msg = """
You are an expert in extracting structured information from legal documents and contracts.
Identify key details such as parties involved, dates, terms, obligations, and legal definitions.
Present the extracted information in a clear, structured format. Be concise, focusing on essential
legal content and ignoring unnecessary boilerplate language."""

In [8]:
def extract(document, model='gpt-4.1', temperature=0):
    response = client.beta.chat.completions.parse(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": document}
        ],
        response_format=Contract
    )

    return json.loads(response.choices[0].message.content)

### CUAD Dataset

We will use a single text document from the Contract Understanding Atticus Dataset (CUAD) for demonstration purposes.

In [9]:
with open("./data/license_agreement.txt", "r") as f:
    document = f.read()

In [10]:
print(document[:500])

1

                                                                 EXHIBIT 10.4

                 LICENSING AND WEB SITE HOSTING AGREEMENT

     This Agreement is entered into on February 26, 1999, (the "Effective Date") by and between Mortgage Logic.com, Inc. ("Client"), with an address at Two Venture Plaza, 2 Venture, Irvine, California 92618 and TrueLink, Inc. ("TrueLink"), with an address at 3026 South Higuera, San Luis Obispo, California 93401.

     WHEREAS, TrueLink is in the business of


In [11]:
data = extract(document)
data

{'contract_type': 'Licensing Agreement',
 'parties': [{'name': 'Mortgage Logic.com, Inc.',
   'location': {'address': 'Two Venture Plaza, 2 Venture',
    'city': 'Irvine',
    'state': 'California',
    'country': 'US'},
   'role': 'Client'},
  {'name': 'TrueLink, Inc.',
   'location': {'address': '3026 South Higuera',
    'city': 'San Luis Obispo',
    'state': 'California',
    'country': 'US'},
   'role': 'Provider'}],
 'effective_date': '1999-02-26',
 'term': 'Initial term of 1 year from Effective Date; automatically renews for successive one-year periods unless either party gives written notice of non-renewal at least 30 days prior to the end of the then-current term. Sections 2 and 3 may terminate earlier upon certain breaches. Termination by Client requires written notice; services continue until the last day of the month following notice.',
 'contract_scope': 'TrueLink grants Client a non-exclusive license to use the Interface for origination, underwriting, processing, and fund

## Constructing the Graph

Finally, we will import the extracted structured output into Neo4j, which follows the standard approach for importing structured data.

We will first design a suitable graph model that represents the relationships and entities in our data.

Based on the pydantic model we defined for structured output, we can create a corresponding graph model representing a contract system with three main entities: `Contract`, `Organization`, and `Location`.
- The `Contract` node stores details such as its ID, type, effective date, term, total amount, governing law, and scope.
- The `Organization` node is linked to contracts through the `HAS_PARTY` relationship, and each organization has a `HAS_LOCATION` relationship to a `Location` node that captures the organization's address, city, state, and country.
- The `Location` node captures the geographical details of the organization's location, since a single organization can have multiple locations.

For example,

![graph model](./imgs/contract-graph-model.png)

Next step is to implement the graph construction and data import process.

First, we will define unique constraints and indexes to ensure data integrity and improve performance. Then we will import the structured contract data into Neo4j with Cypher statements.

Once the data is loaded, we will visualize the graph to confirm that all entities and relationships are correctly represented.

Finally, we will address important data refinement tasks, such as entity resolution, which ensures that different representations of the same real-world entity are merged correctly.

### Data Import

Defining unique constraints and indexes wherever applicable is a best practice, as it not only ensures the integrity of the graph but also enhances query performance.

The following Cypher statements define unique constraints for `Contract`, `Organization`, and `Location` nodes:

In [ ]:
neo4j_driver.execute_query(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Contract) REQUIRE c.id IS UNIQUE;"
)
neo4j_driver.execute_query(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (o:Organization) REQUIRE o.name IS UNIQUE;"
)
neo4j_driver.execute_query(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (l:Location) REQUIRE l.fullAddress IS UNIQUE;"
)

Next, we need to prepare an import Cypher statement that will take the dictionary output and load it into Neo4j, adhering to the graph schema we designed.

In [ ]:
import_query = """WITH $data AS contract_data
// Create Contract node
MERGE (contract:Contract {id: randomUUID()})
SET contract += {
  contract_type: contract_data.contract_type,
  effective_date: contract_data.effective_date,
  term: contract_data.term,
  contract_scope: contract_data.contract_scope,
  end_date: contract_data.end_date,
  total_amount: contract_data.total_amount,
  governing_law: contract_data.governing_law.state + ' ' +
                 contract_data.governing_law.country
}
WITH contract, contract_data
// Create Party nodes and their locations
UNWIND contract_data.parties AS party
MERGE (p:Organization {name: party.name})
MERGE (loc:Location {
  fullAddress: party.location.address + ' ' +
                party.location.city + ' ' +
                party.location.state + ' ' +
                party.location.country})
SET loc += {
  address: party.location.address,
  city: party.location.city,
  state: party.location.state,
  country: party.location.country
}
// Link party to their location
MERGE (p)-[:LOCATED_AT]->(loc)
// Link parties to the contract
MERGE (p)-[r:HAS_PARTY]->(contract)
SET r.role = party.role
"""

In [ ]:
neo4j_driver.execute_query(import_query, data=data)

For example, the final visualization of the graph after importing the data should look like this:

![graph visualization](./imgs/graph-data-example.png)

### Entity Resolution

Entity resolution refers to the process of identifying and merging different representations of the same real-world entity within a data set or knowledge graph.

For example, we may have names:
- UTI Asset Management Company
- UTI Asset Management Company Limited
- UTI Asset Management Company Ltd

Entity resolution in this context involves identifying that all these variations refer to the same real-world organization, despite minor differences in naming conventions (such as “Limited” vs. “Ltd”).

The goal of entity resolution is to unify these disparate references into a single, coherent node within the graph.

Techniques used in entity resolution include string matching, clustering algorithms, and even machine learning methods that use the context surrounding each entity to detect and resolve duplicates.

### Adding Unstructured Data to the Graph

Storing the original unstructured documents and the extracted structured data within the graph preserves the richness of the original data while enabling more precise querying and analysis of the extracted information.

For example, our expanded graph schema where structured and unstructured information is combined may look like this:

![expanded graph schema](./imgs/expanded-graph-model.png)

When incorporating unstructured data into a graph, it is common to use a simple chunking strategy based on token count or word length to split text into manageable segments.